# 02 · An agent that uses a remote MCP server

The three SQL tools now live in the MCP server `shop` (`app/shop_server.py`, no authentication). Any MCP host
(Claude, Cursor, Claude Code) and this agent use the same tools through one URL.

**Before you run this notebook**, start the server in a terminal opened in the session folder (keep it running):
```bash
uv run app/shop_server.py --http            # http://localhost:8000/mcp
uv run app/shop_server.py --http --public   # plus a public ngrok URL, for Claude
```
The server terminal logs its tool calls. How to connect Claude, Cursor and Claude Code: `docs/02_remote_mcp.md`.

<img src="images/b3_mcp_flow.png" width="640" alt="one question through host, model and MCP server">

Docs: [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture) ·
[LangChain MCP](https://docs.langchain.com/oss/python/langchain/mcp)

In [2]:
import json
import logging
import os
import time
import warnings

import httpx
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core._api import LangChainBetaWarning

os.environ["FASTMCP_LOG_ENABLED"] = "false"  # before fastmcp is imported: no rich handler of its own,
                                             # its log lines go through the format below
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2", "mcp", "fastmcp"):  # HTTP and MCP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
warnings.filterwarnings("ignore", category=LangChainBetaWarning)  # langchain.mcp is in beta

# ── Keys and model (.env of this session folder) ─────────────────────────────
load_dotenv()
for name in ("OPENAI_API_KEY", "TAVILY_API_KEY"):
    if not os.getenv(name, "").strip():
        raise RuntimeError(f"Missing {name}: copy .env.example to .env in the session folder and fill it in")
model = init_chat_model(os.getenv("MODEL", "openai:gpt-6-luna"))
logger.info("Model: %s", model.model_name)

# SHOP_MCP_URL (optional): the --public URL; by default the local server on SHOP_PORT
URL = os.getenv("SHOP_MCP_URL") or f"http://localhost:{os.getenv('SHOP_PORT', '8000')}/mcp"


def check_server(url: str, start: str) -> None:
    """Fail fast with the command to run when the MCP server is not up."""
    try:
        httpx.get(url.removesuffix("/mcp") + "/healthz", timeout=3).raise_for_status()
    except httpx.HTTPError:
        raise RuntimeError(f"No MCP server at {url}. Start it in a terminal (session folder): {start}") from None
    logger.info("shop server: ok")  # no URL in the output: it depends on this machine's port


check_server(URL, "uv run app/shop_server.py --http")

INFO | Model: gpt-6-luna
INFO | shop server: ok


## 1. Tools
Two **local** tools from Session 02 (web search and reading) and the **MCP** tools of the `shop` server.
`MCPAdapter` wraps a FastMCP `Client`: it lists the server's tools and turns each one into a LangChain tool.

In [3]:
# ── Local tools: web search and page reading (copied from Session 02) ───────
from langchain.tools import tool
from tavily import BadRequestError, TavilyClient

tavily = TavilyClient()  # uses TAVILY_API_KEY


def log_call(tool: str, args: str, status: str, started: float) -> None:
    """The one log line of a tool call: tool(short args) -> ok|error <ms> ms."""
    args = " ".join(args.split())
    logger.info("%s(%s) -> %s %d ms", tool, args if len(args) <= 70 else args[:70] + "…", status,
                (time.monotonic() - started) * 1000)


@tool
def search_docs(query: str) -> list[dict]:
    """Search the web for documents about the query. Returns the title and url of the top 3 results."""
    started, status = time.monotonic(), "error"
    try:
        hits = tavily.search(query, max_results=3)["results"]
        status = "ok"
        return [{"title": h["title"], "url": h["url"]} for h in hits]
    finally:
        log_call("search_docs", repr(query), status, started)


@tool
def read_doc(url: str, question: str) -> str:
    """Read the parts of one document (a web page from search_docs) that answer the question."""
    started, status = time.monotonic(), "error"
    try:
        pages = tavily.extract(urls=[url], query=question, chunks_per_source=3)["results"]
        if not pages:
            return f"ERROR: could not read {url}"
        status = "ok"
        return pages[0]["raw_content"]
    except BadRequestError as error:  # bad input: tell the model so it can pick another page
        return f"ERROR: could not read {url} ({error})"
    finally:
        log_call("read_doc", repr(url), status, started)


# ── MCP tools: the shop server ───────────────────────────────────────────────
from fastmcp import Client
from langchain.mcp import MCPAdapter

async with MCPAdapter(Client(URL)) as shop:  # connect, list the tools, disconnect
    shop_tools = await shop.list_tools()     # each tool reconnects when the agent calls it
for t in shop_tools:
    logger.info("MCP tool %-15s %s", t.name, t.description.splitlines()[0])

INFO | MCP tool list_tables     List the tables of the shop database. Call it first, then describe_table.
INFO | MCP tool describe_table  Show the columns of one table and 3 sample rows. Call it before writing SQL on that table.
INFO | MCP tool run_query       Run one read-only SQLite SELECT on the shop database and return at most 50 rows.


**Any MCP server, no custom code.** `MCPAdapter` also takes the standard `mcpServers` dict, the same JSON as
Claude Desktop (`claude_desktop_config.json`), Claude Code (`.mcp.json`) and Cursor (`mcp.json`). Adding a server is
one more entry: `{"url": ...}`, `{"url": ..., "headers": {"Authorization": "Bearer ..."}}` for an API key, or
`{"command": "uv", "args": [...]}` for a local stdio server. With two or more servers the adapter prefixes each
tool with its key (`shop_run_query`), so they never clash; one server keeps the plain names (below). Docs: [Multiple servers](https://docs.langchain.com/oss/python/langchain/mcp/connections#multiple-servers) ·
[MCP JSON configuration](https://gofastmcp.com/integrations/mcp-json-configuration)

In [4]:
config = {"mcpServers": {"shop": {"url": URL}}}  # pasted as is from a Claude / Cursor config
async with MCPAdapter(config) as mcp:
    logger.info("Tools from mcpServers: %s", [t.name for t in await mcp.list_tools()])

INFO | Tools from mcpServers: ['list_tables', 'describe_table', 'run_query']


## 2. Agent
Local and MCP tools go into one list: the model does not know where a tool runs.

**Logging vs streaming.** *Logging* is diagnostics for the developer: lines your code and the servers write with
`logger.info` (`INFO | …`), kept or dropped by level. *Streaming* is the agent's own output delivered to the user while
it is produced: `stream()` uses `agent.astream(..., stream_mode=["updates", "messages"])`, prints one line per tool
call (`→`) and tool result (`←`) from `updates`, and the answer token by token from `messages`, instead of waiting
silently for the whole run. Docs: [Streaming](https://docs.langchain.com/oss/python/langchain/streaming)

In [5]:
from langchain.agents import create_agent
from langchain.messages import AIMessageChunk, ToolMessage

agent = create_agent(
    model,
    tools=[search_docs, read_doc, *shop_tools],
    system_prompt=("You are a data analyst for an online store. Use the shop database for the store's numbers "
                   "(amounts in VND, exclude cancelled orders from revenue) and the web only for outside context. "
                   "Answer briefly, with the numbers, and cite web urls."),
)


def short(text: str, n: int = 110) -> str:
    return text if len(text) <= n else text[:n] + "…"


async def stream(agent, question: str) -> list:
    """Stream one run: a line per tool call and tool result ("updates"), the answer token by token ("messages").
    Returns the messages the run added."""
    added, mid_line = [], False
    async for chunk in agent.astream({"messages": [{"role": "user", "content": question}]},
                                     stream_mode=["updates", "messages"], version="v2"):
        if chunk["type"] == "messages":  # (token, metadata): print the model's text as it is generated
            token, meta = chunk["data"]
            if isinstance(token, AIMessageChunk) and meta["langgraph_node"] == "model" and token.text:
                print(token.text, end="", flush=True)
                mid_line = True
            continue
        for update in chunk["data"].values():  # updates: {node: what the node added to the state}
            for m in (update or {}).get("messages", []):
                added.append(m)
                lines = [f"→ {c['name']}({short(json.dumps(c['args'], ensure_ascii=False))})"
                         for c in getattr(m, "tool_calls", [])]
                if isinstance(m, ToolMessage):
                    lines.append(f"← {m.name} {m.status}: {short(m.text)}")
                if lines:
                    print(("\n" if mid_line else "") + "\n".join(lines), flush=True)
                    mid_line = False
    print()
    return added

## 3. Demo
Watch the server terminal next to the notebook: the MCP tool calls arrive there while the answer streams here.

The demo is one turn, so the question is a plain string: edit it and run the cell again. Reading the user's input
(`input()`, a chat UI) is only needed when the agent talks back: multi-turn, resume, human-in-the-loop (Session 02).

In [6]:
question = "Which month of 2026 had the most orders, and which city ordered the most in that month?"
messages = await stream(agent, question)

→ list_tables({})
← list_tables success: ["customers","order_items","orders","products"]
→ describe_table({"table": "orders"})
← describe_table success: {"table":"orders","columns":[{"name":"id","type":"INTEGER"},{"name":"customer_id","type":"INTEGER"},{"name":"s…
→ describe_table({"table": "customers"})
← describe_table success: {"table":"customers","columns":[{"name":"id","type":"INTEGER"},{"name":"name","type":"TEXT"},{"name":"email","…
→ run_query({"sql": "SELECT strftime('%Y-%m', created_at) AS month, COUNT(*) AS total_orders, SUM(CASE WHEN status <> 'can…)
← run_query success: {"rows":[{"month":"2026-09","total_orders":210,"non_cancelled_orders":197},{"month":"2026-05","total_orders":2…
→ run_query({"sql": "SELECT c.city, COUNT(*) AS total_orders, SUM(CASE WHEN o.status <> 'cancelled' THEN 1 ELSE 0 END) AS …)
← run_query success: {"rows":[{"city":"TP. Hồ Chí Minh","total_orders":75,"non_cancelled_orders":71},{"city":"Hà Nội","total_orders…
**September 2026** had the most orders: 